In [1]:
from parser import Parser
import shapely
from shapely.ops import transform
import osmnx as ox
import geopandas as gpd
import pathlib
import os

In [2]:
save_dir = pathlib.Path("/mnt/c/Users/nikita/qgisData/busroutes")
save_dir.mkdir(parents=True, exist_ok=True)

In [3]:
with open("input/boundaries.geojson", "r") as f:
    geojson = f.read()
boundaries = shapely.from_geojson(geojson)
graph = ox.graph_from_polygon(boundaries, network_type="drive", simplify=False)
nodes, edges = ox.graph_to_gdfs(graph)

In [4]:
bus_parser = Parser.BusGraphParser("Санкт-Петербург")
example_route = bus_parser.get_route("/spb/bus/61")

example_route = [(y, x) for x, y in example_route]

read https://kudikina.ru/spb/bus/61/map from cache


In [5]:
example_point = example_route[0]
example_point = shapely.geometry.Point(example_point[0], example_point[1])
with open(save_dir / "example_point.geojson", "w") as f:
    f.write(shapely.to_geojson(example_point))

In [6]:
def find_near_points(point, gdf, tolerance=0.01):
    buffer = point.buffer(tolerance)
    buffer_gdf = gpd.GeoDataFrame(geometry=[buffer], crs=gdf.crs)
    within_points = gpd.sjoin(gdf, buffer_gdf, predicate="within")
    return within_points

In [7]:
near_points = find_near_points(example_point, nodes, tolerance=0.005)

In [8]:
l = shapely.LineString(example_route)

In [ ]:
with open(save_dir / "boundaries.geojson", "w") as f:
    f.write(shapely.to_geojson(boundaries))
with open(save_dir / "example_route.geojson", "w") as f:
    f.write(shapely.to_geojson(l))
with open(save_dir / "points.geojson", "w") as f:
    f.write(near_points.to_json())


if "edges.geojson" not in os.listdir(save_dir):
    with open(save_dir / "edges.geojson", "w") as f:
        f.write(edges.to_json())

if "nodes.geojson" not in os.listdir(save_dir):
    with open(save_dir / "nodes.geojson", "w") as f:
        f.write(nodes.to_json())

# near_points.to_file(save_dir/"points.geojson", driver="GeoJSON")
# edges.to_file(save_dir/"edges.geojson", driver="GeoJSON")
# nodes.to_file(save_dir/"nodes.geojson", driver="GeoJSON")